# 01 — EDA Forense Benner: Gestão de Processos Jurídicos

Notebook para a nova base Benner do jurídico, com aproximadamente 193.950 registros.

Objetivos:
1. Validar grão e chaves da base.
2. Auditar nulos, cardinalidade, valores dominantes e datas.
3. Mapear variáveis jurídicas, administrativas e financeiras.
4. Identificar possíveis variáveis de leakage para modelos futuros.
5. Aprofundar a análise do `Valorajuizado`.
6. Descobrir quais tipos de causa/carteira/produto/assunto/fase concentram maior exposição financeira.
7. Criar relatórios para orientar a priorização do jurídico.

Principais colunas esperadas, com base no dicionário enviado:
`OGC_FID`, `Identificador`, `Numerodistribuicao`, `Situacao`, `Data`, `Datainicial`, `Carteira`, `Cpf_Autor`, `Autor`, `Credenciado`, `Nm_Produto`, `Adv_Interno`, `Nm_Empresa`, `Dataencerramento`, `Motivo_Encerramento`, `Denominacao`, `Nm_Acao`, `Nm_Assunto`, `Valorajuizado`, `Processorelevante`, `Localdeservico`, `Nrprocessoantigo`, `Uf`, `Sub_Motivo`, `Comarca`, `Qtddias_Ultimoandamento`, `Tipoprocesso`, `Dt_Cad2`, `Numerodocontrato`, `Datadocontrato`, `Passiveldeacordo`, `Advogadoadverso`, `Fasedoprocesso`, `Estimativa_De_Perda`.

## 1. Imports e configuração

In [ ]:
import pandas as pd
import numpy as np
import re
import unicodedata
from pathlib import Path

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 200)

BASE_DIR = Path("..")  # altere para Path('.') se o notebook estiver na raiz
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
REPORTS_DIR = BASE_DIR / "outputs" / "reports"

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
REPORTS_DIR.mkdir(parents=True, exist_ok=True)

DATA_FILE = RAW_DIR / "benner_processos.parquet"
# Se estiver em CSV:
# DATA_FILE = RAW_DIR / "benner_processos.csv"

TECH_ID_COL = "OGC_FID"
INTERNAL_PROCESS_ID_COL = "Identificador"
JUDICIAL_PROCESS_ID_COL = "Numerodistribuicao"
SITUATION_COL = "Situacao"
MAIN_DATE_COL = "Data"
START_DATE_COL = "Datainicial"
END_DATE_COL = "Dataencerramento"
PORTFOLIO_COL = "Carteira"
AUTHOR_DOC_COL = "Cpf_Autor"
AUTHOR_COL = "Autor"
EXTERNAL_LAWYER_COL = "Credenciado"
PRODUCT_COL = "Nm_Produto"
INTERNAL_LAWYER_COL = "Adv_Interno"
COMPANY_COL = "Nm_Empresa"
CLOSING_REASON_COL = "Motivo_Encerramento"
COURT_UNIT_COL = "Denominacao"
ACTION_COL = "Nm_Acao"
SUBJECT_COL = "Nm_Assunto"
VALUE_COL = "Valorajuizado"
RELEVANT_PROCESS_COL = "Processorelevante"
SERVICE_AREA_COL = "Localdeservico"
OLD_PROCESS_COL = "Nrprocessoantigo"
UF_COL = "Uf"
SUB_REASON_COL = "Sub_Motivo"
DISTRICT_COL = "Comarca"
DAYS_SINCE_LAST_MOVE_COL = "Qtddias_Ultimoandamento"
PROCESS_TYPE_COL = "Tipoprocesso"
SECONDARY_DATE_COL = "Dt_Cad2"
CONTRACT_NUMBER_COL = "Numerodocontrato"
CONTRACT_DATE_COL = "Datadocontrato"
SETTLEMENT_ELIGIBLE_COL = "Passiveldeacordo"
OPPOSING_LAWYER_COL = "Advogadoadverso"
PROCESS_PHASE_COL = "Fasedoprocesso"
LOSS_ESTIMATE_COL = "Estimativa_De_Perda"

print(DATA_FILE)

## 2. Funções utilitárias

In [ ]:
NULL_STRINGS = {"", "null", "nan", "none", "na", "n/a", "-", "não informado", "nao informado", "sem informação", "sem informacao"}

def read_data(path, columns=None, nrows=None):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")
    if path.suffix.lower() == ".parquet":
        return pd.read_parquet(path, columns=columns)
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, usecols=columns, nrows=nrows)
    if path.suffix.lower() in [".xlsx", ".xls"]:
        return pd.read_excel(path, usecols=columns, nrows=nrows)
    raise ValueError(f"Formato não suportado: {path}")

def normalize_text(text):
    if pd.isna(text):
        return None
    text = str(text).strip().lower()
    text = unicodedata.normalize("NFKD", text)
    text = "".join(c for c in text if not unicodedata.combining(c))
    text = re.sub(r"[^a-z0-9\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    if text in NULL_STRINGS:
        return None
    return text

def is_null_like_series(s):
    if s.dtype == "object" or str(s.dtype) == "category":
        return s.isna() | s.astype(str).str.strip().str.lower().isin(NULL_STRINGS)
    return s.isna()

def existing_cols(df, cols):
    return [c for c in cols if c in df.columns]

def save_report(df, filename):
    path = REPORTS_DIR / filename
    df.to_csv(path, index=False)
    print(f"Salvo: {path}")

## 3. Carregar base e validar colunas

In [ ]:
df = read_data(DATA_FILE)
print("Shape:", df.shape)
df.head()

In [ ]:
expected_cols = [
    TECH_ID_COL, INTERNAL_PROCESS_ID_COL, JUDICIAL_PROCESS_ID_COL, SITUATION_COL,
    MAIN_DATE_COL, START_DATE_COL, PORTFOLIO_COL, AUTHOR_DOC_COL, AUTHOR_COL,
    EXTERNAL_LAWYER_COL, PRODUCT_COL, INTERNAL_LAWYER_COL, COMPANY_COL, END_DATE_COL,
    CLOSING_REASON_COL, COURT_UNIT_COL, ACTION_COL, SUBJECT_COL, VALUE_COL,
    RELEVANT_PROCESS_COL, SERVICE_AREA_COL, OLD_PROCESS_COL, UF_COL, SUB_REASON_COL,
    DISTRICT_COL, DAYS_SINCE_LAST_MOVE_COL, PROCESS_TYPE_COL, SECONDARY_DATE_COL,
    CONTRACT_NUMBER_COL, CONTRACT_DATE_COL, SETTLEMENT_ELIGIBLE_COL,
    OPPOSING_LAWYER_COL, PROCESS_PHASE_COL, LOSS_ESTIMATE_COL
]
col_validation = pd.DataFrame({
    "coluna_esperada": expected_cols,
    "existe_na_base": [c in df.columns for c in expected_cols]
})
save_report(col_validation, "column_validation_expected_benner.csv")
col_validation

## 4. Overview estrutural

Roteiro de explicação: nesta etapa verificamos o tamanho da base, número de colunas, memória e duplicatas completas. Isso ajuda a confirmar se o volume recebido é compatível com os ~193.950 registros informados e se há duplicidade bruta.

In [ ]:
overview = pd.DataFrame([{
    "linhas": len(df),
    "colunas": df.shape[1],
    "memoria_mb": round(df.memory_usage(deep=True).sum() / 1024**2, 2),
    "duplicatas_completas": int(df.duplicated().sum())
}])
save_report(overview, "overview_benner_processos.csv")
overview

In [ ]:
df.dtypes.value_counts().to_frame("qtd_colunas")

## 5. Auditoria de grão e chaves

O dicionário sugere três chaves diferentes:
- `OGC_FID`: identificador técnico da linha.
- `Identificador`: código interno do processo no sistema legado.
- `Numerodistribuicao`: número judicial/distribuição.

A pergunta é: a base está em uma linha por processo interno, por processo judicial, por contrato ou por registro administrativo?

In [ ]:
key_cols = existing_cols(df, [TECH_ID_COL, INTERNAL_PROCESS_ID_COL, JUDICIAL_PROCESS_ID_COL, OLD_PROCESS_COL, CONTRACT_NUMBER_COL])
key_audit = []
for col in key_cols:
    nunique = df[col].nunique(dropna=True)
    key_audit.append({
        "coluna": col,
        "qtd_linhas": len(df),
        "qtd_unicos": nunique,
        "qtd_nulos": int(df[col].isna().sum()),
        "perc_nulos": float(df[col].isna().mean()),
        "linhas_por_chave_media": len(df) / nunique if nunique else np.nan,
        "qtd_duplicados_alem_unicos": len(df) - nunique
    })
key_audit = pd.DataFrame(key_audit).sort_values("qtd_unicos", ascending=False)
save_report(key_audit, "key_audit_benner.csv")
key_audit

In [ ]:
if INTERNAL_PROCESS_ID_COL in df.columns:
    dup_identificador = (
        df.groupby(INTERNAL_PROCESS_ID_COL, dropna=False)
        .size()
        .reset_index(name="qtd_linhas")
        .query("qtd_linhas > 1")
        .sort_values("qtd_linhas", ascending=False)
    )
    save_report(dup_identificador, "duplicidade_identificador.csv")
    display(dup_identificador.head(20))
else:
    print("Identificador não encontrado.")

## 6. Nulos, cardinalidade e valores dominantes

Roteiro de explicação:
- Nulos mostram o que é aproveitável ou precisa virar flag.
- Cardinalidade indica quais variáveis são categóricas simples, IDs ou alta cardinalidade.
- Valor dominante mostra campos quase constantes.

In [ ]:
null_rows = []
for col in df.columns:
    null_mask = is_null_like_series(df[col])
    p = float(null_mask.mean())
    null_rows.append({
        "coluna": col,
        "dtype": str(df[col].dtype),
        "qtd_nulos": int(null_mask.sum()),
        "perc_nulos": p,
        "qtd_nao_nulos": int((~null_mask).sum()),
        "faixa_nulos": "critico_95_mais" if p >= .95 else "alto_80_95" if p >= .80 else "medio_50_80" if p >= .50 else "baixo_20_50" if p >= .20 else "baixo_menos_20"
    })
null_report = pd.DataFrame(null_rows).sort_values("perc_nulos", ascending=False)
save_report(null_report, "null_report_benner_processos.csv")
null_report.head(100)

In [ ]:
cardinality_report = pd.DataFrame([
    {
        "coluna": col,
        "dtype": str(df[col].dtype),
        "qtd_distintos": int(df[col].nunique(dropna=True)),
        "perc_distintos": float(df[col].nunique(dropna=True) / len(df))
    }
    for col in df.columns
]).sort_values("qtd_distintos", ascending=False)
save_report(cardinality_report, "cardinality_report_benner_processos.csv")
cardinality_report.head(100)

In [ ]:
dominant_rows = []
for col in df.columns:
    vc = df[col].value_counts(dropna=False)
    if len(vc):
        dominant_rows.append({
            "coluna": col,
            "dtype": str(df[col].dtype),
            "valor_mais_frequente": vc.index[0],
            "qtd_valor_mais_frequente": int(vc.iloc[0]),
            "perc_valor_mais_frequente": float(vc.iloc[0] / len(df))
        })
dominant_report = pd.DataFrame(dominant_rows).sort_values("perc_valor_mais_frequente", ascending=False)
save_report(dominant_report, "dominant_report_benner_processos.csv")
dominant_report.head(100)

## 7. Classificação semântica das colunas

In [ ]:
def classify_benner_column(col):
    c = col.lower()
    if c in ["ogc_fid"]:
        return "id_tecnico"
    if "identificador" in c or "numero" in c or "nrprocesso" in c or "contrato" in c or "cpf" in c:
        return "identificacao_chave"
    if "valor" in c:
        return "valor_exposicao"
    if "data" in c or c.startswith("dt_"):
        return "data"
    if "situacao" in c or "fase" in c or "ultimoandamento" in c:
        return "andamento_status"
    if "motivo" in c or "encerramento" in c:
        return "encerramento_resultado"
    if "estimativa" in c or "perda" in c:
        return "provisao_perda"
    if "acordo" in c:
        return "acordo"
    if "produto" in c or "carteira" in c or "empresa" in c or "localdeservico" in c:
        return "negocio_carteira"
    if "autor" in c or "advogado" in c or "credenciado" in c or "adv_interno" in c:
        return "partes_advogados"
    if "acao" in c or "assunto" in c or "tipoprocesso" in c:
        return "classe_assunto_tipo"
    if "uf" in c or "comarca" in c or "denominacao" in c:
        return "territorio_orgao"
    if "relevante" in c:
        return "priorizacao"
    return "outros"

column_groups = pd.DataFrame({"coluna": df.columns})
column_groups["grupo_semantico"] = column_groups["coluna"].apply(classify_benner_column)
save_report(column_groups, "column_groups_benner.csv")
column_groups["grupo_semantico"].value_counts()

In [ ]:
profile = (
    null_report
    .merge(cardinality_report[["coluna", "qtd_distintos", "perc_distintos"]], on="coluna", how="left")
    .merge(dominant_report[["coluna", "valor_mais_frequente", "perc_valor_mais_frequente"]], on="coluna", how="left")
    .merge(column_groups, on="coluna", how="left")
    .sort_values(["grupo_semantico", "perc_nulos"], ascending=[True, False])
)
save_report(profile, "profile_completo_benner_processos.csv")
profile.head(150)

## 8. Candidatas a leakage

Para modelos futuros, cuidado com variáveis que parecem resultado, encerramento ou decisão humana posterior:
`Situacao`, `Dataencerramento`, `Motivo_Encerramento`, `Sub_Motivo`, `Estimativa_De_Perda`, `Processorelevante`, `Passiveldeacordo`.

Elas podem ser usadas para EDA e priorização, mas talvez não como feature, dependendo do objetivo temporal do modelo.

In [ ]:
def identify_benner_leakage_candidates(cols):
    patterns = ["encerramento", "sub_motivo", "motivo", "estimativa_de_perda", "dataencerramento", "situacao"]
    return [c for c in cols if any(p in c.lower() for p in patterns)]

leakage_candidates = pd.DataFrame({"coluna": identify_benner_leakage_candidates(df.columns)})
save_report(leakage_candidates, "leakage_candidates_benner.csv")
leakage_candidates

## 9. Padronização dos campos categóricos principais

Vamos criar colunas `_norm` para facilitar agrupamentos consistentes.

In [ ]:
categorical_business_cols = existing_cols(df, [
    SITUATION_COL, PORTFOLIO_COL, PRODUCT_COL, ACTION_COL, SUBJECT_COL,
    PROCESS_TYPE_COL, UF_COL, DISTRICT_COL, PROCESS_PHASE_COL, LOSS_ESTIMATE_COL,
    SETTLEMENT_ELIGIBLE_COL, RELEVANT_PROCESS_COL, EXTERNAL_LAWYER_COL,
    INTERNAL_LAWYER_COL, OPPOSING_LAWYER_COL, SERVICE_AREA_COL, COURT_UNIT_COL, COMPANY_COL
])
df_eda = df.copy()
for col in categorical_business_cols:
    df_eda[f"{col}_norm"] = df_eda[col].apply(normalize_text)
categorical_business_cols

In [ ]:
for col in categorical_business_cols:
    print("\n" + "="*100)
    print(col)
    display(df_eda[col].value_counts(dropna=False).head(20).reset_index())

## 10. Datas e ciclo de vida

Analisamos data de entrada, cadastro, contrato e encerramento. Também criamos idade do processo, duração até encerramento e flags de inatividade.

In [ ]:
date_cols = existing_cols(df_eda, [MAIN_DATE_COL, START_DATE_COL, END_DATE_COL, SECONDARY_DATE_COL, CONTRACT_DATE_COL])
for col in date_cols:
    df_eda[col] = pd.to_datetime(df_eda[col], errors="coerce")

date_summary = []
for col in date_cols:
    s = df_eda[col]
    date_summary.append({
        "coluna": col,
        "qtd_nao_nulos": int(s.notna().sum()),
        "perc_nao_nulos": float(s.notna().mean()),
        "min_data": s.min(),
        "max_data": s.max(),
        "qtd_nulos": int(s.isna().sum())
    })
date_summary = pd.DataFrame(date_summary).sort_values("perc_nao_nulos", ascending=False)
save_report(date_summary, "date_summary_benner.csv")
date_summary

In [ ]:
candidate_ref_dates = []
for col in [MAIN_DATE_COL, END_DATE_COL, SECONDARY_DATE_COL]:
    if col in df_eda.columns:
        mx = df_eda[col].max()
        if pd.notna(mx):
            candidate_ref_dates.append(mx)
DATA_REF = max(candidate_ref_dates) if candidate_ref_dates else pd.Timestamp.today().normalize()
print("DATA_REF:", DATA_REF)

if START_DATE_COL in df_eda.columns:
    df_eda["idade_processo_dias"] = (DATA_REF - df_eda[START_DATE_COL]).dt.days
    df_eda["idade_processo_anos"] = df_eda["idade_processo_dias"] / 365.25
    df_eda["ano_inicio"] = df_eda[START_DATE_COL].dt.year
    df_eda["mes_inicio"] = df_eda[START_DATE_COL].dt.month
    df_eda["ano_mes_inicio"] = df_eda[START_DATE_COL].dt.to_period("M").astype(str)

if END_DATE_COL in df_eda.columns and START_DATE_COL in df_eda.columns:
    df_eda["duracao_ate_encerramento_dias"] = (df_eda[END_DATE_COL] - df_eda[START_DATE_COL]).dt.days

if DAYS_SINCE_LAST_MOVE_COL in df_eda.columns:
    df_eda[DAYS_SINCE_LAST_MOVE_COL] = pd.to_numeric(df_eda[DAYS_SINCE_LAST_MOVE_COL], errors="coerce")
    df_eda["flag_sem_andamento_90d"] = (df_eda[DAYS_SINCE_LAST_MOVE_COL] >= 90).astype(int)
    df_eda["flag_sem_andamento_180d"] = (df_eda[DAYS_SINCE_LAST_MOVE_COL] >= 180).astype(int)
    df_eda["flag_sem_andamento_360d"] = (df_eda[DAYS_SINCE_LAST_MOVE_COL] >= 360).astype(int)

df_eda[[c for c in ["idade_processo_dias", "idade_processo_anos", "duracao_ate_encerramento_dias", "flag_sem_andamento_90d", "flag_sem_andamento_180d", "flag_sem_andamento_360d"] if c in df_eda.columns]].describe()

## 11. Análise aprofundada do Valorajuizado

Pergunta de negócio:
**Quais tipos de causa fazem mais sentido atacar para reduzir perdas financeiras?**

A prioridade não deve ser só volume. Devemos olhar:
- volume;
- valor total ajuizado;
- valor médio/mediano;
- concentração financeira;
- estimativa de perda;
- fase;
- passível de acordo.

In [ ]:
if VALUE_COL not in df_eda.columns:
    raise KeyError(f"Coluna de valor não encontrada: {VALUE_COL}")

df_eda[VALUE_COL] = pd.to_numeric(df_eda[VALUE_COL], errors="coerce")
df_eda["valor_ajuizado"] = df_eda[VALUE_COL]
df_eda["flag_valor_ajuizado_nulo"] = df_eda["valor_ajuizado"].isna().astype(int)
df_eda["flag_valor_ajuizado_zero_ou_negativo"] = (df_eda["valor_ajuizado"].fillna(0) <= 0).astype(int)
df_eda["valor_ajuizado_log1p"] = np.where(df_eda["valor_ajuizado"].fillna(0) > 0, np.log1p(df_eda["valor_ajuizado"]), 0)

valor_summary = df_eda["valor_ajuizado"].describe(percentiles=[.01,.05,.10,.25,.50,.75,.90,.95,.99])
valor_summary.to_frame("valor_ajuizado").to_csv(REPORTS_DIR / "valor_ajuizado_summary.csv")
valor_summary

### Pareto do valor ajuizado

In [ ]:
df_val = df_eda[df_eda["valor_ajuizado"].notna() & (df_eda["valor_ajuizado"] > 0)].copy()
df_val = df_val.sort_values("valor_ajuizado", ascending=False)
df_val["valor_acumulado"] = df_val["valor_ajuizado"].cumsum()
df_val["rank_percentual"] = np.arange(1, len(df_val)+1) / len(df_val)

pareto = []
total_val = df_val["valor_ajuizado"].sum()
for pct in [.01, .05, .10, .20, .30]:
    mask = df_val["rank_percentual"] <= pct
    pareto.append({
        "top_percentual_processos": pct,
        "qtd_processos": int(mask.sum()),
        "valor_total_concentrado": float(df_val.loc[mask, "valor_ajuizado"].sum()),
        "perc_valor_total_concentrado": float(df_val.loc[mask, "valor_ajuizado"].sum() / total_val)
    })
pareto = pd.DataFrame(pareto)
save_report(pareto, "pareto_valor_ajuizado.csv")
pareto

### Faixas de valor ajuizado

In [ ]:
bins = [-np.inf,0,1000,5000,10000,25000,50000,100000,250000,500000,1000000,np.inf]
labels = ["00_sem_valor_ou_zero","01_ate_1k","02_1k_5k","03_5k_10k","04_10k_25k","05_25k_50k","06_50k_100k","07_100k_250k","08_250k_500k","09_500k_1mm","10_acima_1mm"]

df_eda["faixa_valor_ajuizado"] = pd.cut(df_eda["valor_ajuizado"].fillna(0), bins=bins, labels=labels, include_lowest=True).astype(str)

faixa_valor = (
    df_eda.groupby("faixa_valor_ajuizado")
    .agg(
        qtd_processos=(df_eda.columns[0], "count"),
        valor_total=("valor_ajuizado", "sum"),
        valor_medio=("valor_ajuizado", "mean"),
        valor_mediano=("valor_ajuizado", "median")
    )
    .reset_index()
)
faixa_valor["perc_processos"] = faixa_valor["qtd_processos"] / faixa_valor["qtd_processos"].sum()
faixa_valor["perc_valor_total"] = faixa_valor["valor_total"] / faixa_valor["valor_total"].sum()
save_report(faixa_valor, "distribuicao_faixa_valor_ajuizado.csv")
faixa_valor

## 12. Estimativa de perda como eixo de risco

`Estimativa_De_Perda` pode ser usada para priorização e target futuro. Ela não deve ser usada como feature se o modelo tentar prevê-la.

In [ ]:
def map_loss_estimate(value):
    t = normalize_text(value)
    if t is None:
        return np.nan
    if "remoto" in t:
        return 0
    if "possivel" in t:
        return 1
    if "provavel" in t:
        return 2
    return np.nan

if LOSS_ESTIMATE_COL in df_eda.columns:
    df_eda["estimativa_perda_score"] = df_eda[LOSS_ESTIMATE_COL].apply(map_loss_estimate)
    df_eda["flag_perda_provavel"] = (df_eda["estimativa_perda_score"] == 2).astype(int)
    df_eda["flag_perda_possivel_ou_provavel"] = (df_eda["estimativa_perda_score"] >= 1).astype(int)

    loss_dist = (
        df_eda.groupby(LOSS_ESTIMATE_COL, dropna=False)
        .agg(
            qtd_processos=(df_eda.columns[0], "count"),
            valor_total=("valor_ajuizado", "sum"),
            valor_medio=("valor_ajuizado", "mean"),
            valor_mediano=("valor_ajuizado", "median")
        )
        .reset_index()
    )
    loss_dist["perc_processos"] = loss_dist["qtd_processos"] / loss_dist["qtd_processos"].sum()
    loss_dist["perc_valor_total"] = loss_dist["valor_total"] / loss_dist["valor_total"].sum()
    save_report(loss_dist, "distribuicao_estimativa_perda.csv")
    display(loss_dist)
else:
    print("Estimativa_De_Perda não encontrada.")

## 13. Exposição financeira por grupos jurídicos

Agora agregamos por carteira, produto, ação, assunto, tipo de processo, UF, comarca, fase, escritório e advogados.

In [ ]:
if RELEVANT_PROCESS_COL in df_eda.columns:
    df_eda["flag_processo_relevante"] = df_eda[RELEVANT_PROCESS_COL].apply(lambda x: 1 if normalize_text(x) in ["sim","s","yes"] else 0)
else:
    df_eda["flag_processo_relevante"] = 0

def financial_by_group(df, group_col, min_count=30):
    if group_col not in df.columns:
        return None
    agg = {
        "qtd_processos": (df.columns[0], "count"),
        "valor_total": ("valor_ajuizado", "sum"),
        "valor_medio": ("valor_ajuizado", "mean"),
        "valor_mediano": ("valor_ajuizado", "median"),
        "valor_p90": ("valor_ajuizado", lambda x: x.quantile(.90)),
        "valor_p95": ("valor_ajuizado", lambda x: x.quantile(.95)),
        "qtd_processos_relevantes": ("flag_processo_relevante", "sum"),
    }
    if "flag_perda_provavel" in df.columns:
        agg["qtd_perda_provavel"] = ("flag_perda_provavel", "sum")
        agg["taxa_perda_provavel"] = ("flag_perda_provavel", "mean")
    if "flag_perda_possivel_ou_provavel" in df.columns:
        agg["qtd_perda_possivel_ou_provavel"] = ("flag_perda_possivel_ou_provavel", "sum")
        agg["taxa_perda_possivel_ou_provavel"] = ("flag_perda_possivel_ou_provavel", "mean")
    out = df.groupby(group_col, dropna=False).agg(**agg).reset_index()
    out = out[out["qtd_processos"] >= min_count].copy()
    out["share_valor_total"] = out["valor_total"] / out["valor_total"].sum()
    if "taxa_perda_possivel_ou_provavel" in out.columns:
        out["exposicao_risco_proxy"] = out["taxa_perda_possivel_ou_provavel"] * out["valor_total"]
        out["share_exposicao_risco_proxy"] = out["exposicao_risco_proxy"] / out["exposicao_risco_proxy"].sum()
    else:
        out["exposicao_risco_proxy"] = out["valor_total"]
        out["share_exposicao_risco_proxy"] = out["share_valor_total"]
    return out.sort_values("exposicao_risco_proxy", ascending=False)

group_cols = existing_cols(df_eda, [
    PORTFOLIO_COL, PRODUCT_COL, ACTION_COL, SUBJECT_COL, PROCESS_TYPE_COL, UF_COL, DISTRICT_COL,
    PROCESS_PHASE_COL, EXTERNAL_LAWYER_COL, INTERNAL_LAWYER_COL, OPPOSING_LAWYER_COL,
    SERVICE_AREA_COL, COURT_UNIT_COL, COMPANY_COL, SETTLEMENT_ELIGIBLE_COL, RELEVANT_PROCESS_COL
])

financial_reports = {}
for col in group_cols:
    report = financial_by_group(df_eda, col, min_count=30)
    if report is not None:
        financial_reports[col] = report
        safe = re.sub(r"[^a-zA-Z0-9_]", "_", col)
        save_report(report, f"financial_risk_by_{safe}.csv")
        print("\n" + "="*100)
        print(col)
        display(report.head(20))

## 14. Priorização por assunto e ação + assunto

A ideia é classificar os tipos de causa entre:
- prioridade 1: alto volume + alta exposição + alta perda possível/provável;
- prioridade 2: alta exposição em massa;
- prioridade 3: poucos casos de alto valor;
- prioridade 4: alto volume e alta taxa, mesmo com valor menor;
- monitorar.

In [ ]:
if SUBJECT_COL in financial_reports:
    subject_priority = financial_reports[SUBJECT_COL].copy()
    volume_median = subject_priority["qtd_processos"].median()
    risk_median = subject_priority["exposicao_risco_proxy"].median()
    rate_median = subject_priority["taxa_perda_possivel_ou_provavel"].median() if "taxa_perda_possivel_ou_provavel" in subject_priority.columns else np.nan

    subject_priority["volume_alto"] = subject_priority["qtd_processos"] >= volume_median
    subject_priority["risco_financeiro_alto"] = subject_priority["exposicao_risco_proxy"] >= risk_median
    subject_priority["taxa_perda_alta"] = subject_priority["taxa_perda_possivel_ou_provavel"] >= rate_median if "taxa_perda_possivel_ou_provavel" in subject_priority.columns else False

    def classify_priority(row):
        if row["risco_financeiro_alto"] and row["volume_alto"] and row["taxa_perda_alta"]:
            return "prioridade_1_atacar_agora"
        if row["risco_financeiro_alto"] and row["volume_alto"]:
            return "prioridade_2_alta_exposicao_em_massa"
        if row["risco_financeiro_alto"] and not row["volume_alto"]:
            return "prioridade_3_casos_alto_valor"
        if row["volume_alto"] and row["taxa_perda_alta"]:
            return "prioridade_4_melhoria_tese_ou_operacao"
        return "monitorar"

    subject_priority["prioridade_financeira"] = subject_priority.apply(classify_priority, axis=1)
    save_report(subject_priority, "financial_priority_by_assunto_benner.csv")
    display(subject_priority.head(50))
else:
    print("Relatório por assunto não disponível.")

In [ ]:
if ACTION_COL in df_eda.columns and SUBJECT_COL in df_eda.columns:
    df_eda["acao_assunto"] = df_eda[ACTION_COL].astype(str).fillna("NAO_INFORMADO") + " | " + df_eda[SUBJECT_COL].astype(str).fillna("NAO_INFORMADO")
    action_subject_report = financial_by_group(df_eda, "acao_assunto", min_count=20)
    save_report(action_subject_report, "financial_risk_by_acao_assunto.csv")
    display(action_subject_report.head(50))

## 15. Passível de acordo e processo relevante

Essas variáveis são importantes para priorização operacional. Um caso pode ser valioso para ataque rápido se:
- tem valor alto;
- está marcado como passível de acordo;
- está em perda possível/provável;
- está em fase favorável para negociação.

In [ ]:
if SETTLEMENT_ELIGIBLE_COL in df_eda.columns:
    settlement_report = financial_by_group(df_eda, SETTLEMENT_ELIGIBLE_COL, min_count=1)
    save_report(settlement_report, "settlement_eligible_financial_report.csv")
    display(settlement_report)

In [ ]:
if RELEVANT_PROCESS_COL in df_eda.columns:
    relevant_report = financial_by_group(df_eda, RELEVANT_PROCESS_COL, min_count=1)
    save_report(relevant_report, "processo_relevante_financial_report.csv")
    display(relevant_report)

## 16. Ranking financeiro de processos

Criamos um ranking usando `valor_ajuizado × peso_estimativa_perda`.

Pesos exploratórios:
- Remoto = 0.10
- Possível = 0.50
- Provável = 0.90

Esses pesos não são oficiais; servem como proxy para priorização inicial.

In [ ]:
def loss_estimate_weight(value):
    t = normalize_text(value)
    if t is None:
        return 0.30
    if "remoto" in t:
        return 0.10
    if "possivel" in t:
        return 0.50
    if "provavel" in t:
        return 0.90
    return 0.30

if LOSS_ESTIMATE_COL in df_eda.columns:
    df_eda["peso_estimativa_perda"] = df_eda[LOSS_ESTIMATE_COL].apply(loss_estimate_weight)
else:
    df_eda["peso_estimativa_perda"] = 0.30

df_eda["exposicao_risco_proxy"] = df_eda["valor_ajuizado"].fillna(0) * df_eda["peso_estimativa_perda"]

ranking_cols = existing_cols(df_eda, [
    INTERNAL_PROCESS_ID_COL, JUDICIAL_PROCESS_ID_COL, SITUATION_COL, PORTFOLIO_COL, PRODUCT_COL,
    ACTION_COL, SUBJECT_COL, PROCESS_TYPE_COL, UF_COL, DISTRICT_COL, COURT_UNIT_COL,
    PROCESS_PHASE_COL, SETTLEMENT_ELIGIBLE_COL, RELEVANT_PROCESS_COL, LOSS_ESTIMATE_COL,
    VALUE_COL, "valor_ajuizado", "faixa_valor_ajuizado", "peso_estimativa_perda", "exposicao_risco_proxy"
])
ranking_processos = df_eda[ranking_cols].sort_values("exposicao_risco_proxy", ascending=False)
save_report(ranking_processos, "ranking_financeiro_processos_benner.csv")
ranking_processos.head(100)

## 17. Recomendações de ação por assunto

In [ ]:
if "subject_priority" in globals():
    def recommend_action(row):
        prioridade = row["prioridade_financeira"]
        if prioridade == "prioridade_1_atacar_agora":
            return "Revisar tese jurídica, causa raiz operacional e estratégia de acordo/defesa prioritária"
        if prioridade == "prioridade_2_alta_exposicao_em_massa":
            return "Automatizar triagem, padronizar defesa e avaliar acordo em lote"
        if prioridade == "prioridade_3_casos_alto_valor":
            return "Criar governança executiva e acompanhamento individual dos casos"
        if prioridade == "prioridade_4_melhoria_tese_ou_operacao":
            return "Investigar documentação, qualidade das provas e origem operacional da demanda"
        return "Monitorar e reavaliar no próximo ciclo"

    subject_priority["acao_recomendada"] = subject_priority.apply(recommend_action, axis=1)
    save_report(subject_priority, "financial_action_by_assunto_benner.csv")
    display(subject_priority.head(50))

## 18. Resumo executivo financeiro

In [ ]:
executive_summary = {
    "qtd_registros": len(df_eda),
    "qtd_com_valor_ajuizado": int(df_eda["valor_ajuizado"].notna().sum()),
    "perc_com_valor_ajuizado": float(df_eda["valor_ajuizado"].notna().mean()),
    "valor_total_ajuizado": float(df_eda["valor_ajuizado"].sum()),
    "valor_medio_ajuizado": float(df_eda["valor_ajuizado"].mean()),
    "valor_mediano_ajuizado": float(df_eda["valor_ajuizado"].median()),
    "valor_p90_ajuizado": float(df_eda["valor_ajuizado"].quantile(.90)),
    "valor_p95_ajuizado": float(df_eda["valor_ajuizado"].quantile(.95)),
    "valor_p99_ajuizado": float(df_eda["valor_ajuizado"].quantile(.99)),
    "exposicao_risco_proxy_total": float(df_eda["exposicao_risco_proxy"].sum()),
}
if "flag_perda_provavel" in df_eda.columns:
    executive_summary["qtd_perda_provavel"] = int(df_eda["flag_perda_provavel"].sum())
    executive_summary["taxa_perda_provavel"] = float(df_eda["flag_perda_provavel"].mean())
if "flag_perda_possivel_ou_provavel" in df_eda.columns:
    executive_summary["qtd_perda_possivel_ou_provavel"] = int(df_eda["flag_perda_possivel_ou_provavel"].sum())
    executive_summary["taxa_perda_possivel_ou_provavel"] = float(df_eda["flag_perda_possivel_ou_provavel"].mean())

executive_summary_df = pd.DataFrame([executive_summary])
save_report(executive_summary_df, "executive_summary_financeiro_benner.csv")
executive_summary_df

## 19. Salvar base analítica da EDA

In [ ]:
df_eda.to_parquet(PROCESSED_DIR / "benner_processos_eda_analitica.parquet", index=False)
print("Base salva em:", PROCESSED_DIR / "benner_processos_eda_analitica.parquet")
print(df_eda.shape)

## 20. Validar saídas esperadas

In [ ]:
expected_outputs = [
    REPORTS_DIR / "overview_benner_processos.csv",
    REPORTS_DIR / "key_audit_benner.csv",
    REPORTS_DIR / "null_report_benner_processos.csv",
    REPORTS_DIR / "cardinality_report_benner_processos.csv",
    REPORTS_DIR / "profile_completo_benner_processos.csv",
    REPORTS_DIR / "leakage_candidates_benner.csv",
    REPORTS_DIR / "valor_ajuizado_summary.csv",
    REPORTS_DIR / "pareto_valor_ajuizado.csv",
    REPORTS_DIR / "distribuicao_faixa_valor_ajuizado.csv",
    REPORTS_DIR / "distribuicao_estimativa_perda.csv",
    REPORTS_DIR / "ranking_financeiro_processos_benner.csv",
    REPORTS_DIR / "executive_summary_financeiro_benner.csv",
    PROCESSED_DIR / "benner_processos_eda_analitica.parquet",
]
for path in expected_outputs:
    print(path, "OK" if path.exists() else "NÃO ENCONTRADO")

# Roteiro de explicação da EDA

1. **Grão e chaves**: validamos se a base está em uma linha por processo interno, número judicial, contrato ou registro administrativo.
2. **Qualidade dos dados**: avaliamos nulos, cardinalidade e valores dominantes.
3. **Grupos semânticos**: classificamos variáveis em identificação, carteira, produto, assunto, ação, território, advogados, fase, encerramento, acordo e perda.
4. **Leakage**: marcamos campos que podem representar informação posterior, como encerramento, motivo, submotivo e estimativa de perda.
5. **Valor ajuizado**: investigamos distribuição, cauda longa, faixas e concentração financeira.
6. **Exposição por grupos**: avaliamos carteira, produto, assunto, ação, UF, comarca, fase, escritório e advogados.
7. **Estimativa de perda**: usamos como eixo de risco/provisão para priorizar a exposição financeira.
8. **Priorização**: criamos matriz por assunto e ranking de processos com maior exposição de risco proxy.
9. **Ação**: geramos recomendação por assunto: atacar agora, alta exposição em massa, casos de alto valor, melhoria de tese/operação ou monitorar.

Próximos notebooks sugeridos:
- `02_feature_engineering_benner_processos.ipynb`
- `03_modelagem_estimativa_perda_benner.ipynb`
- `04_modelagem_priorizacao_financeira_benner.ipynb`